In [1]:
import sys
import os

print("python:", sys.executable)
print("Exixts:", os.path.exists(sys.executable))

python: c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe
Exixts: True


In [2]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)

c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.version)

3.5.6


In [4]:
# ============================================================
# QUESTION 1 — CUSTOMER TRANSACTION SUMMARY
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# An e-commerce company receives transaction records from multiple stores.
# The analytics team wants a customer-level transaction summary.
 

transactions = sc.parallelize([
    ("T001", "C101", 1200.0),
    ("T002", "C102", 500.0),
    ("T003", "C101", 800.0),
    ("T004", "C103", 1500.0),
    ("T005", "C102", 700.0),
    ("T006", "C101", 1000.0),
    ("T007", "C104", 400.0),
    ("T008", "C103", 500.0)
], 4)


In [7]:
pair_rdd = transactions.map(lambda x:(x[1],(x[2],1)))
result = pair_rdd.reduceByKey(lambda x,y:(x[0]+y[0],x[1]+x[1]))
mapValues = result.mapValues(lambda x:(x[0],x[1],x[0]/x[1]))
mapValues.collect()

[('C103', (2000.0, 2, 1000.0)),
 ('C101', (3000.0, 4, 750.0)),
 ('C104', (400.0, 1, 400.0)),
 ('C102', (1200.0, 2, 600.0))]

In [14]:
# using aggregate by Key
maping = transactions.map(lambda x:(x[1],x[2]))
def add_values(so_far,value):
    return(
        so_far[0]+value,
        so_far[1]+1
    )
    
def combine(tot1,tot2):
    total = tot1[0]+tot2[0]
    count = tot1[1]+tot2[1]
    return(total,count)
    
pair_rdd = maping.aggregateByKey(
    (0,0),
    add_values,
    combine
)      



In [15]:
result = pair_rdd.mapValues(lambda x:(x[0],x[1],x[0]/x[1]))
result.collect()

[('C103', (2000.0, 2, 1000.0)),
 ('C101', (3000.0, 3, 1000.0)),
 ('C104', (400.0, 1, 400.0)),
 ('C102', (1200.0, 2, 600.0))]

In [16]:
# ============================================================
# QUESTION 2 — FAILED TRANSACTION DETECTION
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# A payment company wants customers having repeated failed transactions.

transactions = sc.parallelize([
    ("T001", "C101", "SUCCESS"),
    ("T002", "C102", "FAILED"),
    ("T003", "C101", "FAILED"),
    ("T004", "C103", "SUCCESS"),
    ("T005", "C102", "FAILED"),
    ("T006", "C101", "FAILED"),
    ("T007", "C104", "FAILED"),
    ("T008", "C102", "SUCCESS"),
    ("T009", "C105", "FAILED"),
    ("T010", "C105", "FAILED")
])


In [20]:
filtering = transactions.filter(lambda x:x[2]=="FAILED").map(lambda x:(x[1],1))
result = filtering.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1] >1)
result.collect()



[('C102', 2), ('C105', 2), ('C101', 2)]

In [22]:
# ============================================================
# QUESTION 3 — DUPLICATE TRANSACTION DETECTION
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# Because of retries from an upstream system, some transaction IDs were received multiple times.

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "C102", 700),
    ("T001", "C101", 500),
    ("T003", "C103", 900),
    ("T004", "C104", 200),
    ("T002", "C102", 700),
    ("T002", "C102", 700),
    ("T005", "C105", 1000)
])

# Requirement:
# Find duplicate transaction IDs and occurrence count.


In [24]:
filtering = transactions.map(lambda x:(x[0],1))
result = filtering.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1] >1)
result.collect()

[('T001', 2), ('T002', 3)]

In [26]:
# ============================================================
# QUESTION 4 — LATEST CUSTOMER TRANSACTION
# LEVEL: MEDIUM-HARD
# ============================================================

# Scenario:
# Customers can perform multiple transactions.
# Business wants only the latest transaction for every customer.

transactions = sc.parallelize([
    ("C101", "T001", "2026-08-20 10:00:00", 500),
    ("C102", "T002", "2026-08-20 11:00:00", 700),
    ("C101", "T003", "2026-08-21 09:00:00", 900),
    ("C103", "T004", "2026-08-20 12:30:00", 400),
    ("C102", "T005", "2026-08-22 10:15:00", 1200),
    ("C101", "T006", "2026-08-22 15:00:00", 1500)
])


In [27]:
maping = transactions.map(lambda x:(x[0],(x[1],x[2],x[3])))
result = maping.reduceByKey(lambda x,y: x if x[1] > y[1] else y)
result.collect()

[('C102', ('T005', '2026-08-22 10:15:00', 1200)),
 ('C103', ('T004', '2026-08-20 12:30:00', 400)),
 ('C101', ('T006', '2026-08-22 15:00:00', 1500))]

In [34]:
# ============================================================
# QUESTION 5 — PRODUCT REVENUE
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# An online retailer wants total revenue generated by every product.

sales = sc.parallelize([
    ("T001", "Laptop", 2, 60000),
    ("T002", "Mouse", 5, 500),
    ("T003", "Laptop", 1, 60000),
    ("T004", "Keyboard", 3, 1500),
    ("T005", "Mouse", 10, 500),
    ("T006", "Monitor", 2, 12000),
    ("T007", "Keyboard", 2, 1500)
])


In [46]:
maping = sales.map(lambda x:(x[1],x[2]*x[3])).reduceByKey(lambda x,y:x+y)
maping.collect()

    

[('Laptop', 180000), ('Monitor', 24000), ('Mouse', 7500), ('Keyboard', 7500)]

In [49]:
# ============================================================
# QUESTION 6 — CUSTOMER TRANSACTION ENRICHMENT
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# Customer master and transaction data exist separately.
# Transactions must be enriched with customer information.

customers = sc.parallelize([
    ("C101", ("Anuj", "Mumbai")),
    ("C102", ("Rahul", "Delhi")),
    ("C103", ("Priya", "Pune")),
    ("C104", ("Neha", "Bangalore"))
])

transactions = sc.parallelize([
    ("C101", ("T001", 500)),
    ("C102", ("T002", 700)),
    ("C101", ("T003", 900)),
    ("C103", ("T004", 400))
])


In [50]:
result = customers.join(transactions)
result.collect()

[('C103', (('Priya', 'Pune'), ('T004', 400))),
 ('C101', (('Anuj', 'Mumbai'), ('T001', 500))),
 ('C101', (('Anuj', 'Mumbai'), ('T003', 900))),
 ('C102', (('Rahul', 'Delhi'), ('T002', 700)))]

In [57]:
def faltten(records):
    cust_id,(names,trans) = records
    
    name,city = names
    tans_id,amount = trans
    
    return [
        ( tans_id,
        cust_id,
        name,
        city,
        amount)
    ]    
    
result1 = result.flatMap(faltten)
result1.collect()     

[('T004', 'C103', 'Priya', 'Pune', 400),
 ('T001', 'C101', 'Anuj', 'Mumbai', 500),
 ('T003', 'C101', 'Anuj', 'Mumbai', 900),
 ('T002', 'C102', 'Rahul', 'Delhi', 700)]

In [70]:
# ============================================================
# QUESTION 7 — CUSTOMERS WITHOUT TRANSACTIONS
# LEVEL: MEDIUM
# ============================================================

customers = sc.parallelize([
    ("C101", "Anuj"),
    ("C102", "Rahul"),
    ("C103", "Priya"),
    ("C104", "Neha"),
    ("C105", "Amit")
])

transactions = sc.parallelize([
    ("C101", 500),
    ("C102", 700),
    ("C101", 900),
    ("C104", 400)
])

# Requirement:
# Find customers having zero transactions.


In [75]:
join = customers.leftOuterJoin(transactions).filter(lambda x:x[1][1] is None)
join.collect()

[('C103', ('Priya', None)), ('C105', ('Amit', None))]

In [88]:
# ============================================================
# QUESTION 8 — TRANSACTION VALIDATION
# LEVEL: MEDIUM
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "", 700),
    ("T003", "C103", -100),
    ("", "C104", 900),
    ("T005", "C105", 1200),
    ("T006", "C106", 0)
])

# Validation Rules:
# transaction_id must not be empty
# customer_id must not be empty
# amount > 0

In [91]:
def validation(records):
    
    trans_id = records[0]
    cust_id = records[1]
    amount = records[2]
    
    if trans_id == "":
        return (trans_id, cust_id, amount, "TRANSACTION_ID_MISSING")

    elif cust_id == "":
        return (trans_id, cust_id, amount, "CUSTOMER_ID_MISSING")

    elif amount <= 0:
        return (trans_id, cust_id, amount, "INVALID_AMOUNT")

    else:
        return (trans_id, cust_id, amount, "Valid")
      

In [92]:
rejected_rdd = (
    transactions
    .map(validation)
    .filter(lambda x: x[3] != "Valid")
)

rejected_rdd.collect()

[('T002', '', 700, 'CUSTOMER_ID_MISSING'),
 ('T003', 'C103', -100, 'INVALID_AMOUNT'),
 ('', 'C104', 900, 'TRANSACTION_ID_MISSING'),
 ('T006', 'C106', 0, 'INVALID_AMOUNT')]

In [93]:
# ============================================================
# QUESTION 10 — MULTIPLE VALIDATION FAILURES
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("", "", -100),
    ("T003", "", 0),
    ("", "C104", 900)
])

# Requirement:
# Return ALL validation errors for each invalid record.


In [94]:
def validation(records):
    trans_id = records[0]
    cust_id = records[1]
    amount = records[2]
    
    errors = []
    
    if trans_id == "":
        errors.append("Missing_transaction_id")
    if cust_id == "":
        errors.append("Missing_customer_id")   
    if amount <= 0:
        errors.append("Missing amount") 
    if errors:
        return(
            trans_id,cust_id,amount,errors
        )    
           
    return None

In [95]:
rejected_rdd = (
    transactions
    .map(validation)
    .filter(lambda x: x is not None)
)

print(rejected_rdd.collect())

[('', '', -100, ['Missing_transaction_id', 'Missing_customer_id', 'Missing amount']), ('T003', '', 0, ['Missing_customer_id', 'Missing amount']), ('', 'C104', 900, ['Missing_transaction_id'])]


In [101]:
# ============================================================
# QUESTION 11 — DAILY REVENUE
# LEVEL: MEDIUM
# ============================================================

transactions = sc.parallelize([
    ("T001", "2026-08-20 10:10:00", 500),
    ("T002", "2026-08-20 11:30:00", 700),
    ("T003", "2026-08-21 09:10:00", 900),
    ("T004", "2026-08-21 12:00:00", 400),
    ("T005", "2026-08-21 15:20:00", 1200),
    ("T006", "2026-08-22 10:00:00", 600)
])

# Requirement:
# Calculate total revenue by transaction date.

In [104]:
pair_rdd = transactions.map(lambda x:(x[1][:10],x[2])).reduceByKey(lambda x,y:x+y)
pair_rdd.collect()

[('2026-08-20', 1200), ('2026-08-21', 2500), ('2026-08-22', 600)]